In [1]:
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 16.7 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# import
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import numpy as numpy
import os
from PIL import Image

from tqdm import tqdm
import wandb
wandb.login()


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: km936 (km936-cornell-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

Load the Dataset

In [4]:
!wget -c http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip

--2026-05-08 14:35:25--  http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip
Resolving data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)... 129.132.52.178, 2001:67c:10ec:36c2::178
Connecting to data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)|129.132.52.178|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip [following]
--2026-05-08 14:35:26--  https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip
Connecting to data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)|129.132.52.178|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3530603713 (3.3G) [application/zip]
Saving to: ‘DIV2K_train_HR.zip’

DIV2K_train_HR.zip  100%[===================>]   3.29G  21.9MB/s    in 2m 37s  

2026-05-08 14:38:03 (21.4 MB/s) - ‘DIV2K_train_HR.zip’ saved [3530603713/3530603713]



In [5]:
!unzip DIV2K_train_HR.zip

Archive:  DIV2K_train_HR.zip
   creating: DIV2K_train_HR/
  inflating: DIV2K_train_HR/0103.png  
  inflating: DIV2K_train_HR/0413.png  
  inflating: DIV2K_train_HR/0031.png  
  inflating: DIV2K_train_HR/0660.png  
  inflating: DIV2K_train_HR/0126.png  
  inflating: DIV2K_train_HR/0793.png  
  inflating: DIV2K_train_HR/0764.png  
  inflating: DIV2K_train_HR/0550.png  
  inflating: DIV2K_train_HR/0437.png  
  inflating: DIV2K_train_HR/0374.png  
  inflating: DIV2K_train_HR/0755.png  
  inflating: DIV2K_train_HR/0614.png  
  inflating: DIV2K_train_HR/0646.png  
  inflating: DIV2K_train_HR/0371.png  
  inflating: DIV2K_train_HR/0312.png  
  inflating: DIV2K_train_HR/0108.png  
  inflating: DIV2K_train_HR/0556.png  
  inflating: DIV2K_train_HR/0794.png  
  inflating: DIV2K_train_HR/0722.png  
  inflating: DIV2K_train_HR/0780.png  
  inflating: DIV2K_train_HR/0555.png  
  inflating: DIV2K_train_HR/0439.png  
  inflating: DIV2K_train_HR/0396.png  
  inflating: DIV2K_train_HR/0666.png  
  infl

In [6]:
!rm -r DIV2K_train_HR.zip

In [7]:
!wget -c http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip

--2026-05-08 14:38:36--  http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip
Resolving data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)... 129.132.52.178, 2001:67c:10ec:36c2::178
Connecting to data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)|129.132.52.178|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip [following]
--2026-05-08 14:38:37--  https://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip
Connecting to data.vision.ee.ethz.ch (data.vision.ee.ethz.ch)|129.132.52.178|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 448993893 (428M) [application/zip]
Saving to: ‘DIV2K_valid_HR.zip’

DIV2K_valid_HR.zip  100%[===================>] 428.19M  21.4MB/s    in 21s     

2026-05-08 14:38:58 (20.4 MB/s) - ‘DIV2K_valid_HR.zip’ saved [448993893/448993893]



In [8]:
!unzip DIV2K_valid_HR.zip

Archive:  DIV2K_valid_HR.zip
   creating: DIV2K_valid_HR/
  inflating: DIV2K_valid_HR/0897.png  
  inflating: DIV2K_valid_HR/0887.png  
  inflating: DIV2K_valid_HR/0806.png  
  inflating: DIV2K_valid_HR/0834.png  
  inflating: DIV2K_valid_HR/0896.png  
  inflating: DIV2K_valid_HR/0881.png  
  inflating: DIV2K_valid_HR/0828.png  
  inflating: DIV2K_valid_HR/0833.png  
  inflating: DIV2K_valid_HR/0877.png  
  inflating: DIV2K_valid_HR/0826.png  
  inflating: DIV2K_valid_HR/0879.png  
  inflating: DIV2K_valid_HR/0812.png  
  inflating: DIV2K_valid_HR/0809.png  
  inflating: DIV2K_valid_HR/0865.png  
  inflating: DIV2K_valid_HR/0882.png  
  inflating: DIV2K_valid_HR/0830.png  
  inflating: DIV2K_valid_HR/0892.png  
  inflating: DIV2K_valid_HR/0859.png  
  inflating: DIV2K_valid_HR/0858.png  
  inflating: DIV2K_valid_HR/0816.png  
  inflating: DIV2K_valid_HR/0836.png  
  inflating: DIV2K_valid_HR/0857.png  
  inflating: DIV2K_valid_HR/0824.png  
  inflating: DIV2K_valid_HR/0823.png  
  infl

In [9]:
!rm -r DIV2K_valid_HR.zip

Dataset Settings

In [10]:
# check where dataset is loaded relative to colab files
root_path_to_image_train = '/content/DIV2K_train_HR'
root_path_to_image_valid = '/content/DIV2K_valid_HR'

# set the batch size and workers here
batch_size = 16
num_workers = 0

In [11]:
# Set up dataset

class Div2KDataset(Dataset):
    def __init__(self, root, transforms=None):
        self.path_to_img = []
        for f in os.listdir(root):
            if f.endswith('png'):
                self.path_to_img.append(os.path.join(root, f))

        self.transforms = transforms

    def __len__(self):
        return len(self.path_to_img)

    def __getitem__(self, idx):
        image = Image.open(self.path_to_img[idx]).convert('RGB')
        if self.transforms:
            image = self.transforms(image)
        return image

In [12]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(360, pad_if_needed=True),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])
#converts each pixel from [0,1] to [-1,1]

validation_transform = transforms.Compose([
    transforms.CenterCrop(360), # keep crop deterministic
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]) # same normalization as train
])

train_dataset = Div2KDataset(root_path_to_image_train, transforms=train_transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)

valid_dataset = Div2KDataset(root_path_to_image_valid, transforms=validation_transform)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

In [13]:
# Check dataset sizes
print(f"Train samples: {len(train_dataset)}")   # 800
print(f"Valid samples: {len(valid_dataset)}")   # 100

# Check a single image loading correctly
sample = train_dataset[0]
print(f"Sample type: {type(sample)}")
print(f"Sample shape: {sample.shape}") # [3, 360, 360]

# Check dataloaders
train_batch = next(iter(train_loader))
print(f"Train batch shape: {train_batch.shape}")  # [batch_size, 3, 360, 360]

valid_batch = next(iter(valid_loader))
print(f"Valid batch shape: {valid_batch.shape}") # [batch_size, 3, 360, 360]

Train samples: 800
Valid samples: 100
Sample type: <class 'torch.Tensor'>
Sample shape: torch.Size([3, 360, 360])
Train batch shape: torch.Size([16, 3, 360, 360])
Valid batch shape: torch.Size([16, 3, 360, 360])


Evaluation Functions

In [14]:
from torchmetrics.image import StructuralSimilarityIndexMeasure

def psnr(cover_image, stego_image):
  """
  Inputs:
  - cover_image: N x 3 x H x W (normalized to [-1, 1])
  - stego_image: N x 3 x H x W (normalized to [-1, 1])

  Returns: scalar average PSNR in dB across the batch
  """
  # pixel range is 2.0 for images in [-1, 1]
  max_val = 2.0
  mse = F.mse_loss(stego_image, cover_image)
  return (10 * torch.log10(max_val ** 2 / mse)).item()

def ssim(cover_image, stego_image):
  """
  Inputs:
  - cover_image: N x 3 x H x W (normalized to [-1, 1])
  - stego_image: N x 3 x H x W (normalized to [-1, 1])

  Returns: scalar SSIM value, closer to 1 is better
  """
  device = cover_image.device
  metric = StructuralSimilarityIndexMeasure(data_range=2.0).to(device)
  return metric(stego_image, cover_image).item()

def evaluate(encoder, decoder, dataloader, D, device="cpu"):
  """
  Inputs:
  - encoder: trained encoder model
  - decoder: trained decoder model
  - dataloader: DataLoader over the validation set
  - D: bits per pixel (must match what encoder/decoder were trained with)
  - device: "cpu" or "cuda"

  Returns: dict with RS-BPP (aggregated over the whole val set) and
           image-weighted average PSNR / SSIM.

  Notes:
  - RS-BPP is computed by summing wrong-bit counts across the entire val set
    and applying the formula once. Per-batch averaging would inflate it
    because of the max(0, ·) clamp (non-linear → mean(max(...)) ≠ max(mean(...))).
  - PSNR / SSIM are weighted by batch size N so the smaller final batch
    (e.g. 4 images out of 100) doesn't get over-weighted.
  """

  #Switch the models from training mode to evaluation mode.
  #BatchNorm behaves differently during evaluation (uses running mean and variance)
  encoder.eval()
  decoder.eval()

  total_wrong = 0
  total_bits  = 0
  psnr_sum = 0.0
  ssim_sum = 0.0
  total_images = 0

  with torch.no_grad():
    for cover_image in dataloader: #cover_image is shape (N, 3, H, W)
      cover_image = cover_image.to(device)
      N, _, H, W = cover_image.shape

      # generate random binary message for this batch
      message = torch.randint(0, 2, (N, D, H, W), dtype=torch.float, device=device)

      # encode and decode
      stego_image = encoder(cover_image, message)
      decoded_message = decoder(stego_image)

      # accumulate raw bit-error counts so we can apply the RS-BPP formula
      # ONCE at the end (avoids the max(0, ·) clamp bias from per-batch averaging)
      predicted_bits = (decoded_message > 0).float()
      total_wrong += (predicted_bits != message).sum().item()
      total_bits  += message.numel()

      # weight image-level metrics by batch size so the trailing partial batch
      # doesn't get over-weighted
      psnr_sum += psnr(cover_image, stego_image) * N
      ssim_sum += ssim(cover_image, stego_image) * N
      total_images += N

  p = total_wrong / total_bits
  rs_bpp = max(0.0, D * (1 - 2 * p))
  acc = 1.0 - (total_wrong / total_bits)

  return {
    "RS-BPP": rs_bpp,
    "PSNR": psnr_sum / total_images,
    "SSIM": ssim_sum / total_images,
    "Acc": acc,
  }


# Qualitative logging helpers (used by the Trainer for wandb image panels)

def _denorm(x):
    # undo image normalization so we can visualize them during wandb logging
    return (x * 0.5 + 0.5).clamp(0, 1)

def _qualitative_samples_dict(encoder, decoder, fixed_batch, D, device, max_images=4):
    """
    Builds (but does NOT log) a wandb-ready dict of qualitative panels for a fixed val batch
    so we can see the same images evolve across epochs. The Trainer merges this dict into
    its eval-metrics log call so all per-epoch values share the same wandb _step.

    Panels:
      - cover:     original images
      - stego:     encoder output given a random message
      - residual:  visual differences between original and encoded image. bright spots show differences
    """

    #switch models to eval mode
    encoder.eval()
    decoder.eval()

    with torch.no_grad():
        cover = fixed_batch.to(device)[:max_images]
        N, _, H, W = cover.shape
        # use a fixed seed to generate the message so that it's the same every epoch for this panel
        g = torch.Generator(device=device).manual_seed(0)
        M = torch.randint(0, 2, (N, D, H, W), generator=g, device=device).float()

        stego = encoder(cover, M)

        cover_v = _denorm(cover)
        stego_v = _denorm(stego)
        #The residual image will show bright spots where the encoder modified the cover image
        #Helps see if the embedding is spatially uniform, and if it hides bits in edgy textured areas or smooth areas
        residual_v = (stego_v - cover_v).abs().mul(10).clamp(0, 1)

        return {
            "val/samples/cover":    [wandb.Image(img) for img in cover_v.cpu()],
            "val/samples/stego":    [wandb.Image(img) for img in stego_v.cpu()],
            "val/samples/residual_x10": [wandb.Image(img) for img in residual_v.cpu()],
        }


Encoders

In [15]:
import torch
from torch import nn

class BasicEncoder(nn.Module):
    def __init__(self, D:int):
        """
        Parameters
        ----------

        D: int
            the number of bits to hide in each pixel of cover image

        """
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, 3,1,"same"),
            nn.LeakyReLU(inplace=True),
            nn.BatchNorm2d(32)
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(32+D, 32, 3,1,"same"),
            nn.LeakyReLU(inplace=True),
            nn.BatchNorm2d(32),

            nn.Conv2d(32, 32, 3,1,"same"),
            nn.LeakyReLU(inplace=True),
            nn.BatchNorm2d(32)
        )
        # omit leaky relu and batch norm for last block
        self.conv3 = nn.Conv2d(32,3,3,1,"same")

    def forward(self, cover_image, message):
        """
        Parameters
        ----------
        cover_image : N x 3 x H x W Tensor
            Cover image C to input to Encoder network, H x W is size of image and 3 RGB channels. N is batch size
        message: {0, 1} N x D x H x W Tensor
            Binary data tensor, secret message M. D is the number of bits to hide in each pixel of cover image.

        Returns
        -------
        output: N x 3 x H x W Tensor
            Output encoded steganographic image S
        """
        x = self.conv1(cover_image)
        x = torch.cat([x, message], dim = 1) # becomes N x 32 + D x H x W
        # pass concatenated input into the next two blocks sequentially
        x = self.conv2(x)
        x = self.conv3(x)
        x = torch.tanh(x) # get back to expected image output range [-1,1] since this predicts entire image
        return x


class ResidualEncoder(nn.Module):
    # def forward(self, cover_image, message):
    #     x = super().forward(cover_image,message)
    #     return cover_image + x

    def __init__(self, D:int):
        """
        Parameters
        ----------

        D: int
            the number of bits to hide in each pixel of cover image

        """
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, 3,1,"same"),
            nn.LeakyReLU(inplace=True),
            nn.BatchNorm2d(32)
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(32+D, 32, 3,1,"same"),
            nn.LeakyReLU(inplace=True),
            nn.BatchNorm2d(32),

            nn.Conv2d(32, 32, 3,1,"same"),
            nn.LeakyReLU(inplace=True),
            nn.BatchNorm2d(32)
        )
        # omit leaky relu and batch norm for last block
        self.conv3 = nn.Conv2d(32,3,3,1,"same")

    def forward(self, cover_image, message):
        """
        Parameters
        ----------
        cover_image : N x 3 x H x W Tensor
            Cover image C to input to Encoder network, H x W is size of image and 3 RGB channels. N is batch size
        message: {0, 1} N x D x H x W Tensor
            Binary data tensor, secret message M. D is the number of bits to hide in each pixel of cover image.

        Returns
        -------
        output: N x 3 x H x W Tensor
            Output encoded steganographic image S
        """
        x = self.conv1(cover_image)
        x = torch.cat([x, message], dim = 1) # becomes N x 32 + D x H x W
        # pass concatenated input into the next two blocks sequentially
        x = self.conv2(x)
        x = self.conv3(x)
        # no Tanh since it now predicts a residual from the input image
        return cover_image + x

class DenseEncoder(nn.Module):
    def __init__(self, D:int):
        """
        Parameters
        ----------

        D: int
            the number of bits to hide in each pixel of cover image

        """
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 32, 3,1,"same"),
            nn.LeakyReLU(inplace=True),
            nn.BatchNorm2d(32)
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(32+D, 32, 3,1,"same"),
            nn.LeakyReLU(inplace=True),
            nn.BatchNorm2d(32)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(32*2 + D, 32, 3,1,"same"),
            nn.LeakyReLU(inplace=True),
            nn.BatchNorm2d(32)
        )
        # omit leaky relu and batch norm for last block
        self.conv4 = nn.Conv2d(32*3 + D,3,3,1,"same")

    def forward(self, cover_image, message):
        """
        Parameters
        ----------
        cover_image : N x 3 x H x W Tensor
            Cover image C to input to Encoder network, H x W is size of image and 3 RGB channels. N is batch size
        message: {0, 1} N x D x H x W Tensor
            Binary data tensor, secret message M. D is the number of bits to hide in each pixel of cover image.

        Returns
        -------
        output: N x 3 x H x W Tensor
            Output encoded steganographic image S
        """
        # list of intermediate outputs
        xs = []
        x = self.conv1(cover_image)
        xs.append(x)
        x = torch.cat(xs + [message], dim = 1)
        x = self.conv2(x)
        xs.append(x)
        x = torch.cat(xs + [message], dim=1)
        x = self.conv3(x)
        xs.append(x)
        x = torch.cat(xs + [message], dim=1)
        x = self.conv4(x)
        # no Tanh since it now predicts a residual from the input image
        return cover_image + x

Decoder

In [16]:
import torch
import torch.nn as nn

class Decoder(nn.Module):
    """
    Spec: See Section 3.2.2 (Equation 6)
    Input Size: (N, 3, H, W)
    Output Size: (N, D, H, W)

    Experiments: Ablate over D={1, 3, 6}
    """

    def __init__(self, D, hidden_dim=32):
        super(Decoder, self).__init__()
        # conv_D-->D' blocks: (page 3, 4 in the paper)
        # (1) Conv2d with in_channel D, out_channel D', kernel size 3, stride 1, padding same (so padding = 1)
        # (2) LeakyRelU activation
        # (3) BatchNormalization
        # omit activation and batch norm if convolution block is last block in network

        self.hidden_dim = hidden_dim
        # cat operation: concat along the depth axis (the channel axis)
        self.a = nn.Conv2d(3, self.hidden_dim, 3, 1, 1)
        self.b = nn.Conv2d(self.hidden_dim, self.hidden_dim, 3, 1, 1)
        self.c = nn.Conv2d(2 * self.hidden_dim, self.hidden_dim, 3, 1, 1) # 64 because we concat a and b which each have 32 channels
        self.d = nn.Conv2d(3 * self.hidden_dim, D, 3, 1, 1) # output channels is D, the hidden_dim
        self.leakyrelu = nn.LeakyReLU(inplace=True)
        self.batchnorm1 = nn.BatchNorm2d(self.hidden_dim) # a, b, c, d have same hidden size (output channel size)
        self.batchnorm2 = nn.BatchNorm2d(self.hidden_dim)
        self.batchnorm3 = nn.BatchNorm2d(self.hidden_dim)

    def forward(self, x):
        a_x = self.batchnorm1(self.leakyrelu(self.a(x)))
        b_x = self.batchnorm2(self.leakyrelu(self.b(a_x)))
        c_x = self.batchnorm3(self.leakyrelu(self.c(torch.concat([a_x, b_x], dim=1)))) # concat along the chanel dim
        d_x = self.d(torch.concat([a_x, b_x, c_x], dim=1))

        return d_x

Training

In [17]:
from torch.nn.utils import clip_grad_norm_
import torch.nn.functional as F
import wandb
from tqdm import tqdm

class Trainer:
    """
    This training class jointly optimizes three networks:
    - Encoder: hides a binary message inside an image
    - Decoder: recovers the hidden message
    - Critic: distinguishes real vs generated images (Wasserstein GAN)

    For each batch:
    1. Sample a random binary message M ~ Ber(0.5)
    2. Update the critic using Wasserstein loss:
    Lc = C(real) - C(fake)
    3. Update encoder + decoder using:
    L = Ld + Ls + Lr
    where:
        Ld: decoding loss (binary cross entropy)
        Ls: similarity loss (MSE between cover and stego image)
        Lr: realism loss from critic

    After each epoch, evaluate using:
        RS-BPP (message capacity)
        PSNR (pixel-level distortion)
        SSIM (perceptual similarity)
    """
    def __init__(self, encoder, decoder, critic, D, device, sample_every=2):
        self.encoder = encoder.to(device)
        self.decoder = decoder.to(device)
        self.critic = critic.to(device)

        self.device = device
        self.D = D

        self.enc_dec_opt = torch.optim.Adam(
            list(self.encoder.parameters()) + list(self.decoder.parameters()),
            lr=1e-4,
        )
        self.critic_opt = torch.optim.Adam(self.critic.parameters(), lr=1e-4)

        self.grad_clip = 0.25
        self.critic_clip = 0.1
        self.sample_every = sample_every  # log image panel every N epochs

    def sample_message(self, N, H, W):
        """
        Inputs:
        - N: batch size
        - D: bits per pixel (data depth)
        - H, W: spatial dimensions
        - device: torch device (cpu or cuda)

        Returns:
        - message: N x D x H x W tensor of binary values {0,1}

        Description:
        - Generates a random binary message for each image in the batch
        - Each pixel stores D bits
        - Values are sampled from a Bernoulli(0.5) distribution
        """
        return torch.randint(0, 2, (N, self.D, H, W), device=self.device).float()

    def similarity_loss(self, cover, generated):
        """
        Inputs:
        - cover: N x 3 x H x W original image
        - generated: N x 3 x H x W stego image

        Returns:
        - scalar similarity loss

        Description:
        - Computes normalized mean squared error between cover and stego images
        - Matches paper formulation:
            Ls = (1 / (3 * H * W)) * ||cover - generated||^2
        - Encourages minimal visual distortion
        """
        _, _, H, W = cover.shape
        return ((cover - generated) ** 2).sum(dim=(1, 2, 3)).mean() / (3 * H * W)

    def train_epoch(self, loader, epoch):

        self.encoder.train()
        self.decoder.train()
        self.critic.train()

        total = {"Lc": 0.0, "Ld": 0.0, "Ls": 0.0, "Lr": 0.0, "L_total": 0.0, "Acc": 0.0}
        steps = 0

        tqdm_bar = tqdm(loader, desc=f"Epoch: {epoch}", leave=True)
        for cover in tqdm_bar:
            cover = cover.to(self.device)
            N, _, H, W = cover.shape

            # sample message to hide for this batch
            M = self.sample_message(N, H, W)

            # Critic
            with torch.no_grad():
                fake = self.encoder(cover, M)

            # Critic Update (Wasserstein GAN + Gradient clipping (stability) + Weight clipping to enforce Lipschitz constraint)
            real_score = self.critic(cover).mean()
            fake_score = self.critic(fake).mean()
            Lc = real_score - fake_score

            self.critic_opt.zero_grad()
            Lc.backward()
            clip_grad_norm_(self.critic.parameters(), self.grad_clip)
            self.critic_opt.step()

            for p in self.critic.parameters():
                p.data.clamp_(-self.critic_clip, self.critic_clip)

            # Encoder-Decoder Loss & Updates
            fake = self.encoder(cover, M)
            decoded = self.decoder(fake)

            # Ld (decoding loss): Binary cross entropy between decoded message and original message
            Ld = F.binary_cross_entropy_with_logits(decoded, M)
            # Ls (similarity loss): MSE between cover and stego image
            Ls = self.similarity_loss(cover, fake)
            # Lr (realness loss): Critic score of generated image
            Lr = self.critic(fake).mean()

            # L_total: combined enc-dec loss that actually drives gradients (paper Eq. 11); ablation: scale Ls by 100 to drive down Ls
            loss = Ld + 100.0 * Ls + Lr # scaling MSE loss

            self.enc_dec_opt.zero_grad()
            loss.backward()
            clip_grad_norm_(
                list(self.encoder.parameters()) + list(self.decoder.parameters()),
                self.grad_clip,
            )
            self.enc_dec_opt.step()

            with torch.no_grad():
                acc = ((decoded >= 0) == (M >= 0.5)).float().mean() # Measures fraction of correctly recovered bits

            total["Lc"] += Lc.item()
            total["Ld"] += Ld.item()
            total["Ls"] += Ls.item()
            total["Lr"] += Lr.item()
            total["L_total"] += loss.item()
            total["Acc"] += acc.item()
            steps += 1

            wandb.log({"train/Lc": Lc.item(), "train/Ld": Ld.item(),
                "train/Ls": Ls.item(), "train/Lr": Lr.item(),
                "train/L_total": loss.item(), "train/Acc": acc.item()})
            tqdm_bar.set_postfix({"Lc": f"{Lc.item():.3f}", "Ld": f"{Ld.item():.3f}", "Acc": f"{acc.item():.3f}"})

        return {k: v / steps for k, v in total.items()}

    def train(self, train_loader, val_loader, epochs):
        # grab ONE fixed val batch up-front so the qualitative panel tracks
        # the same images across all epochs
        fixed_val_batch = next(iter(val_loader))

        for epoch in range(epochs):
            print(f"\nEpoch {epoch+1}")

            train_metrics = self.train_epoch(train_loader, epoch+1)

            print(
                f"Lc: {train_metrics['Lc']:.4f} | "
                f"Ld: {train_metrics['Ld']:.4f} | "
                f"Ls: {train_metrics['Ls']:.6f} | "
                f"Lr: {train_metrics['Lr']:.4f} | "
                f"L_total: {train_metrics['L_total']:.4f} | "
                f"Acc: {train_metrics['Acc']:.4f}"
            )

            # Evaluation metrics, see evaluate.py for details
            eval_metrics = evaluate(self.encoder, self.decoder, val_loader, self.D, device=self.device)

            print(
                f"RS-BPP: {eval_metrics['RS-BPP']:.4f} | "
                f"PSNR: {eval_metrics['PSNR']:.2f} | "
                f"SSIM: {eval_metrics['SSIM']:.4f} | "
                f"Acc: {eval_metrics['Acc']:.4f}"
            )

            # Build a single per-epoch log dict so eval scalars and qualitative
            # images share the same wandb _step (cleaner table view + 1 fewer
            # network call per epoch).
            log_dict = {
                "eval/RS-BPP": eval_metrics["RS-BPP"],
                "eval/PSNR":   eval_metrics["PSNR"],
                "eval/SSIM":   eval_metrics["SSIM"],
                "eval/Acc":    eval_metrics["Acc"],
                "epoch":       epoch + 1,
            }

            # Qualitative image panel on a fixed val batch, every N epochs
            # (also logs on the first epoch so we have a baseline)
            # _qualitative_samples_dict is defined in the Evaluation Functions cell
            if epoch == 0 or (epoch + 1) % self.sample_every == 0 or (epoch + 1) == epochs:
                log_dict.update(_qualitative_samples_dict(
                    self.encoder, self.decoder, fixed_val_batch,
                    D=self.D, device=self.device,
                ))

            wandb.log(log_dict)


Critic

In [18]:
import torch
import torch.nn as nn

class Critic(nn.Module):
    """
    Spec: See Section 3.2.3 (Equation 7)
    Input Size: (N, 3, H, W)
    Output Size: (N, 1, H, W)
    """

    def __init__(self, hidden_dim=32):
        super(Critic, self).__init__()
        # (same as decoder)
        # conv_D-->D' blocks: (page 3, 4 in the paper)
        # (1) Conv2d with in_channel D, out_channel D', kernel size 3, stride 1, padding same (so padding = 1)
        # (2) LeakyRelU activation
        # (3) BatchNormalization
        # omit activation and batch norm if convolution block is last block in network

        self.hidden_dim = hidden_dim
        # cat operation: concat along the depth axis (the channel axis)
        self.a = nn.Conv2d(3, self.hidden_dim, 3, 1, 1) # Conv3->32
        self.b = nn.Conv2d(self.hidden_dim, self.hidden_dim, 3, 1, 1) # Conv32->32
        self.c = nn.Conv2d(self.hidden_dim, self.hidden_dim, 3, 1, 1) # Conv32->32
        self.d = nn.Conv2d(self.hidden_dim, 1, 3, 1, 1) # Conv32->1

        self.leakyrelu = nn.LeakyReLU(inplace=True)
        self.batchnorm1 = nn.BatchNorm2d(self.hidden_dim)
        self.batchnorm2 = nn.BatchNorm2d(self.hidden_dim)
        self.batchnorm3 = nn.BatchNorm2d(self.hidden_dim)


    def forward(self, x):
        a_x = self.batchnorm1(self.leakyrelu(self.a(x)))
        b_x = self.batchnorm2(self.leakyrelu(self.b(a_x)))
        c_x = self.batchnorm3(self.leakyrelu(self.c(b_x)))
        d_x = self.d(c_x) # N x 1 x H x W
        d_x = d_x.mean(dim=[2, 3]).squeeze(1) # find single mean for each channel (mean across H x W dimension --> N x 1 --> (N, ) as a result of squeeze)

        return d_x # dimension: (N, )

Google Drive mounting for checkpoints

In [22]:
SAVE_DIR = "/content/drive/MyDrive/CS4782-Final-Project/steganogan_checkpoints/with_scaling"
os.makedirs(SAVE_DIR, exist_ok=True)

In [23]:
print(SAVE_DIR)
print(os.path.exists(os.path.join(SAVE_DIR, "basic_D1.pt")))

/content/drive/MyDrive/CS4782-Final-Project/steganogan_checkpoints/with_scaling
False


Configs

In [24]:
# Config sweep

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Paper trains: 3 architectures x 6 data depths = 18 configs
# lr=1e-4, grad_clip=0.25, critic_clip=0.1, epochs=32 (all fixed in paper)
EPOCHS   = 32
LR       = 1e-4

encoder_variants = {
    "basic":    BasicEncoder,
    "residual": ResidualEncoder,
    "dense":    DenseEncoder,
}

configs = [
    {"arch": arch_name, "D": D, "label": f"{arch_name}_D{D}"}
    for arch_name in ["residual", "dense"]
    for D in [1, 3, 6]
]

all_results = {}

for cfg in configs:

    checkpoint_path = os.path.join(SAVE_DIR, f"{cfg['label']}.pt")

    # skip + load saved metrics
    if os.path.exists(checkpoint_path):
        print(f"Skipping {cfg['label']} (already trained)")

        ckpt = torch.load(checkpoint_path, map_location=device)

        all_results[cfg["label"]] = {
            **ckpt["metrics"],
            "arch": cfg["arch"],
            "D": cfg["D"],
        }

        continue

    print(f"\n{'='*60}")
    print(f"Running config: {cfg['label']}  (arch={cfg['arch']}, D={cfg['D']})")
    print('='*60)

    # init wandb FIRST so we fail fast (auth/network) before allocating GPU memory.
    wandb.init(
        project="steganogan_metrics",
        name=cfg["label"],
        config={"arch": cfg["arch"], "D": cfg["D"], "epochs": EPOCHS, "lr": LR},
        reinit=True,
    )

    wandb.define_metric("epoch")
    wandb.define_metric("eval/*", step_metric="epoch")
    wandb.define_metric("val/*",  step_metric="epoch")

    EncoderClass = encoder_variants[cfg["arch"]]

    encoder = EncoderClass(D=cfg["D"])
    decoder = Decoder(D=cfg["D"])
    critic  = Critic()

    trainer = Trainer(encoder, decoder, critic, D=cfg["D"], device=device)

    trainer.train(train_loader, valid_loader, epochs=EPOCHS)

    final = evaluate(encoder, decoder, valid_loader, cfg["D"], device=device)

    all_results[cfg["label"]] = {
        **final,
        "arch": cfg["arch"],
        "D": cfg["D"]
    }

    torch.save({
        "encoder_state_dict": encoder.state_dict(),
        "decoder_state_dict": decoder.state_dict(),
        "critic_state_dict": critic.state_dict(),
        "config": cfg,
        "metrics": final,
    }, checkpoint_path)

    print(f"Saved checkpoint to {checkpoint_path}")

    print(
        f"\n[{cfg['label']}] RS-BPP: {final['RS-BPP']:.4f} | "
        f"PSNR: {final['PSNR']:.2f} dB | "
        f"SSIM: {final['SSIM']:.4f} | "
        f"Acc: {final['Acc']:.4f}"
    )

    wandb.finish()

# Summary table (based on paper Table 1)
print(f"\n{'='*72}")
print(f"{'D':<4} {'Arch':<12} {'RS-BPP':>8} {'PSNR':>8} {'SSIM':>8} {'Acc':>8}")
print('-'*72)

for D in [1, 3, 6]:
    for arch in ["basic", "residual", "dense"]:

        label = f"{arch}_D{D}"

        if label not in all_results:
            print(f"D={D}  {arch:<12} MISSING")
            continue

        m = all_results[label]

        print(
            f"D={D}  {arch:<12} "
            f"{m['RS-BPP']:>8.4f} "
            f"{m['PSNR']:>8.2f} "
            f"{m['SSIM']:>8.4f} "
            f"{m['Acc']:>8.4f}"
        )

    print()


Running config: residual_D1  (arch=residual, D=1)


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.



Epoch 1


Epoch: 1: 100%|██████████| 50/50 [02:23<00:00,  2.86s/it, Lc=-0.000, Ld=0.660, Acc=0.640]


Lc: -0.0003 | Ld: 0.6843 | Ls: 0.077292 | Lr: 0.0200 | L_total: 8.4335 | Acc: 0.5841
RS-BPP: 0.2626 | PSNR: 24.13 | SSIM: 0.7049 | Acc: 0.6313

Epoch 2


...

Lc: -0.0264 | Ld: 0.5180 | Ls: 0.000542 | Lr: 0.0059 | L_total: 0.5781 | Acc: 0.6778
RS-BPP: 2.0426 | PSNR: 39.24 | SSIM: 0.9677 | Acc: 0.6702

Epoch 32


Epoch: 32: 100%|██████████| 50/50 [02:38<00:00,  3.18s/it, Lc=-0.024, Ld=0.535, Acc=0.667]


Lc: -0.0274 | Ld: 0.5155 | Ls: 0.000478 | Lr: 0.0063 | L_total: 0.5696 | Acc: 0.6785
RS-BPP: 2.0310 | PSNR: 38.84 | SSIM: 0.9653 | Acc: 0.6692
Saved checkpoint to /content/drive/MyDrive/CS4782-Final-Project/steganogan_checkpoints/with_scaling/dense_D6.pt

[dense_D6] RS-BPP: 2.0323 | PSNR: 38.84 dB | SSIM: 0.9653 | Acc: 0.6694


epoch,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇███
eval/Acc,▁▃▃▄▄▅▅▆▆▇▇▇▇██████████▇█▇▇▇▇▇▇▇
eval/PSNR,▁▃▃▄▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇██████████
eval/RS-BPP,▁▃▃▄▄▅▅▆▆▇▇▇▇██████████▇█▇▇▇▇▇▇▇
eval/SSIM,▁▄▅▆▇▇▇▇████████████████████████
train/Acc,▁▁▄▃▃▃▄▅▆▆▄▅▅▇█▇▆▅▇▆▆▇▇█▇▆▇▆▆▇▆▆▇█▆▆▇▅▆▆
train/L_total,█▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/Lc,█▇▆▅▆▇▇▇▆▆▇▇▆▆▆▆▆▇▅▇▆▆▅▅▆▆▅▄▄▄▂▄▄▄▄▃▁▂▃▄
train/Ld,█▇▇▇▇▆▆▆▆▆▆▆▅▅▅▅▅▄▄▄▃▃▃▃▄▃▃▃▃▁▃▂▃▂▃▁▁▂▂▁
train/Lr,▄▄▄▄▄▅▆▆▇▆▇█▆▆▅▂▂▃▂▂▁▂▂▂▃▄▂▃▃▄▆▆▂▅▂▅▇▆▄█
+1,...



D    Arch           RS-BPP     PSNR     SSIM      Acc
------------------------------------------------------------------------
D=1  basic        MISSING
D=1  residual       0.9773    41.37   0.9800   0.9887
D=1  dense          0.9612    38.77   0.9651   0.9806

D=3  basic        MISSING
D=3  residual       2.1857    37.43   0.9450   0.8643
D=3  dense          2.3224    35.27   0.9279   0.8871

D=6  basic        MISSING
D=6  residual       2.0973    41.92   0.9800   0.6748
D=6  dense          2.0323    38.84   0.9653   0.6694

